# 沪深300未来5日排序：Kaggle 全量运行

这个笔记本只做一件事：用仓库 `https://github.com/xiaohe-4/stock_train` 的固定提交，拉取最新已经收盘的沪深300日线，按原训练配置训练两个 seed，再导出排序。它不是 GPU 检查笔记本。请新建笔记本后导入本文件，不要打开或覆盖账号里已有的私有 GPU 检查笔记本。

本文件不会下单、不会连接券商、不会发消息、也不会把笔记本或数据设为公开。排序是模型分数，不是投资建议。

## 在 Kaggle 界面上先做的设置

1. 用账号 `caihuasheng` 登录 Kaggle。
2. 新建一个笔记本。确认浏览器标签不是已有的 GPU 检查笔记本，再使用 File → Import Notebook 导入 `hs300_5d_rank_kaggle.ipynb`。
3. Accelerator 选 GPU T4 x2。代码只使用 `cuda:0`，第二张 T4 会空闲。如果界面里有单卡 T4，选单卡即可，避免空占第二张卡的配额。
4. Internet 打开。BaoStock 使用 `public-api.baostock.com:10030`，不是普通网页端口。
5. 不要把笔记本设为 Public，不要运行会创建公开 Dataset 的命令。

## 第一个代码单元里的开关

- `RUN_MODE = "full"`：正式排序。使用仓库默认值，不缩短股票池、不减少 epoch、不改特征。
- `RUN_MODE = "smoke"`：只做通路检查。1 个 epoch、只训练 seed 42，写到单独目录。输出明确标成不是排序。
- `RUN_MODE = "preflight"`：只检查代码哈希、数据和 GPU，不训练。
- `RESUME_COMPLETED_SEEDS = False`：默认从头训练两个 seed。只有会话在某个 seed 完整写完 `final_score.txt` 和 `best_model.pth` 后中断，才把它改成 `True` 再跑。这不会把没跑完的 seed 当成完成。
- `USE_OFFLINE_SNAPSHOT = True`：直接读已经挂上的私有数据集，不再下载行情。三个路径已经写死，重新导入后不用改。
- `ALLOW_STALE_SNAPSHOT = False`：快照最大日期必须等于当场算出的完整交易日。

## 仓库默认，full 模式不会改

- 提交：`c735797c86ec3908379f7f54b70953c6a62b23b9`
- 股票池：BaoStock 当前沪深300成分，训练时再按训练集最近 30 日平均成交额保留前 75%
- 特征：`158+39`，序列长度 30
- 标签：`(第 t+5 日开盘 - 第 t+1 日开盘) / 第 t+1 日开盘`
- 损失：TopK-ListMLE，`K=3`，组合收益温度 `0.35`，权重 `0.4`
- 选择指标：`weighted_port_return`
- 集成：seed 42 和 123，分数取平均
- 输出权重：`[0.50, 0.47, 0.01, 0.01, 0.01]`
- 早停耐心：12。早停是原逻辑，不是把训练改短。配置里的 `num_epochs` 仍是 80。

## 为什么推理文件不是最后 5 个交易日

仓库里的 `split_train_test.py` 默认把最后 5 个交易日切成 `test.csv`。推理需要每只股票至少 30 个交易日，技术指标最长看到 60 日。只有 5 行的测试文件不能形成序列。因此本笔记本把同一份完整日线同时写成 `train.csv` 和 `test.csv`。训练脚本仍会在内部把最后约两个月划成验证集。被打分的日期是文件里的最大完整交易日，这一天还没有未来 5 日开盘，不会成为训练标签。

## 这次交付时已经核对的日期

2026-11-10 12:15（Asia/Shanghai）向 BaoStock 重拉了当前沪深300。当天是交易日，但日线还没有 2026-11-10。300 只股票的最后一根完整 K 线都是 **2026-11-09**。按仓库标签，这一天对应的区间是 2026-11-10 开盘到 2026-11-16 开盘。笔记本运行时会重新计算这个上限；如果那时已经收盘且数据源给出了新的完整日线，as-of 日期会跟着变，不会沿用 11 月 9 日。

254 行历史停牌日有价格、没有成交量，都不在 2026-11-09。训练脚本原本就会用训练集中位数填缺失值。

## 完整交易日

A 股当日 15:00（Asia/Shanghai）收盘后，日线才可能完整。11:30 到 13:00 的午间，以及下午盘中，当天都不是完整日线。笔记本用 Asia/Shanghai 的当前时刻和 BaoStock 交易日历计算上限，再核对下载结果的实际最大日期。不会把没有返回的日期补出来。

## 预期资源和时间

准备这个笔记本的云主机没有 GPU，也没有跑出排序。下面的数字分两类：已经在这台机器上量到的，以及还没在 T4 上量到的。

- 下载：BaoStock 单只股票从 2024-01-01 拉到完整交易日大约 6 秒。300 只大约 30 到 45 分钟。
- 特征：在这台机器上，`158+39` 特征列表实际是 196 列，单只 617 行大约 0.06 秒。300 只大约 20 秒，训练和推理各做一次。这不是耗时大头。
- 模型：`StockTransformer` 458,370 个参数。batch 是 8×约 225 只×30 日×196 列。这台 CPU 上一个这样的 batch 前向加反向大约 3.2 秒。若一个 epoch 有大约 80 个 batch、跑满 80 个 epoch、两个 seed，CPU 上大约 11 小时。T4 会比这台 CPU 快，但这里没有 T4，不能把 11 小时写成 Kaggle 时间。
- 在 Kaggle 上用第一个 epoch 校正：剩余时间大约等于「该 epoch 的秒数 × 还没跑的 epoch × 还没跑的 seed」。早停耐心是 12，经常不会跑满 80。若按这个速度两个 seed 会超过 10 小时，停下来保留日志，不要自行把 epoch、股票池或特征改小。
- 磁盘：日线几十 MB，checkpoint 很小。不要重装 Kaggle 已有的 CUDA PyTorch。
- 配额：按一次 4 到 8 小时的 GPU 会话准备。跑完就停。不要并行开第二个 GPU 会话。代码只用第一张卡。

## 跑完后要下载的文件

会话还活着时，从 `/kaggle/working/hs300_rank_output/` 下载：

- `ranking_full.csv`：按模型分数降序，含股票代码、名称、分数、区间和 as-of 日期
- `result.csv`：原项目的前 5 名和权重
- `VERIFY_PASS.txt`：校验通过才会出现。失败时是 `VERIFY_FAIL.txt`
- `run_manifest.json`
- `verification.json`

## 怎样确认这是一次真实排序

同时满足才算数：

1. 日志里的模式是 `full`，没有 `SMOKE TEST ONLY`。
2. `git rev-parse HEAD` 等于 `c735797c86ec3908379f7f54b70953c6a62b23b9`，并且驱动打印源文件哈希通过。
3. 数据最大日期等于本次 Asia/Shanghai 完整交易日上限，文件中没有更晚的日期，最大日期上至少有 290 只股票。
4. `model/30_158+39/seed_42` 和 `seed_123` 都有 `best_model.pth` 与 `final_score.txt`。
5. 两个 seed 的 `config.json` 仍是 80 epoch、`158+39`、序列 30、龙头比例 0.75、权重 `[0.50, 0.47, 0.01, 0.01, 0.01]`。
6. `ranking_full.csv` 按 `model_score` 降序，代码、名称、分数、horizon、`asof_date` 都有值，名称没有空的。
7. `result.csv` 的 5 个代码和权重与排序前 5 行一致。
8. 没有一只被打分的股票，其最后一根 K 线早于 `asof_date`。
9. 存在 `VERIFY_PASS.txt`。

缺少其中任何一条，就还没有正式排序。不要用 smoke 输出或 2026-07-22 的旧仓库数据充数。


In [ ]:
RUN_MODE = "full"  # full | smoke | preflight
RESUME_COMPLETED_SEEDS = False
USE_OFFLINE_SNAPSHOT = True
ALLOW_STALE_SNAPSHOT = False
OFFLINE_PANEL = "/kaggle/input/datasets/caihuasheng/hs300-latest-complete-daily-snapshot/hs300_daily_hfq_20240101_complete.csv"
OFFLINE_NAMES = "/kaggle/input/datasets/caihuasheng/hs300-latest-complete-daily-snapshot/hs300_constituents.csv"
OFFLINE_MANIFEST = "/kaggle/input/datasets/caihuasheng/hs300-latest-complete-daily-snapshot/hs300_data_manifest.json"
PINNED_COMMIT = "c735797c86ec3908379f7f54b70953c6a62b23b9"
REPO_URL = "https://github.com/xiaohe-4/stock_train.git"
print("mode", RUN_MODE)
assert RUN_MODE in {"full", "smoke", "preflight"}
if RUN_MODE != "full":
    print("THIS IS NOT THE FINAL RANKING")
from pathlib import Path
missing = [p for p in (OFFLINE_PANEL, OFFLINE_NAMES, OFFLINE_MANIFEST) if not Path(p).is_file()]
if USE_OFFLINE_SNAPSHOT and missing:
    raise SystemExit("attached snapshot files not found:\n" + "\n".join(missing))


In [ ]:
import base64
import subprocess
import sys
from pathlib import Path

KIT = Path("/kaggle/working/hs300_kaggle_kit")
REPO = Path("/kaggle/working/stock_train")
OUTPUT = Path("/kaggle/working/hs300_rank_output")
KIT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
BLOBS = {"refresh_hs300.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJSZWZyZXNoIENTSSAzMDAgZGFpbHkgYmFycyB3aXRoIHRoZSByZXBvc2l0b3J5J3MgQmFvU3RvY2sgbWV0aG9kLgoKVGhlIGV4ZWN1dGFibGUgcHJvamVjdCB3aW5kb3cgc3RhcnRzIGF0IDIwMjQtMDEtMDEgKHNlZSBUSFUtQkRDMjAyNi1tYWluL2dldF9zdG9ja19kYXRhLnB5KS4KQmFycyB1c2UgYWRqdXN0ZmxhZz0iMSIgKOWQjuWkjeadgykgYW5kIHRoZSBzYW1lIGNvbHVtbiBtYXBwaW5nIGFzIGdldF9zdG9ja19oaXN0b3J5LgoKQW4gdW5maW5pc2hlZCBBLXNoYXJlIHNlc3Npb24gaXMgbm90IGEgY29tcGxldGUgZGFpbHkgYmFyLiBCZWZvcmUgMTU6MDAKQXNpYS9TaGFuZ2hhaSBvbiBhIHRyYWRpbmcgZGF5LCB0aGF0IGNhbGVuZGFyIGRheSBpcyBleGNsdWRlZCBldmVuIGlmIGEgZGF0YQp2ZW5kb3Igd291bGQgZW1pdCBhIHBhcnRpYWwgcm93LiBUaGUgc2F2ZWQgcGFuZWwgaXMgdGhlbiBjaGVja2VkIGFnYWluc3QgdGhlCmxhdGVzdCBkYXRlIHRoYXQgd2FzIGFjdHVhbGx5IHJldHVybmVkLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgc29ja2V0CmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHRyYWNlYmFjawpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZSBhcyBkdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB6b25laW5mbyBpbXBvcnQgWm9uZUluZm8KCmltcG9ydCBwYW5kYXMgYXMgcGQKClNIQU5HSEFJID0gWm9uZUluZm8oIkFzaWEvU2hhbmdoYWkiKQpDTE9TRV9USU1FID0gZHRpbWUoMTUsIDApCkRFRkFVTFRfU1RBUlQgPSAiMjAyNC0wMS0wMSIKUkVRVUlSRURfQ09MVU1OUyA9IFsKICAgICLogqHnpajku6PnoIEiLAogICAgIuaXpeacnyIsCiAgICAi5byA55uYIiwKICAgICLmlLbnm5giLAogICAgIuacgOmrmCIsCiAgICAi5pyA5L2OIiwKICAgICLmiJDkuqTph48iLAogICAgIuaIkOS6pOminSIsCiAgICAi5oyv5bmFIiwKICAgICLmtqjot4zpop0iLAogICAgIuaNouaJi+eOhyIsCiAgICAi5rao6LeM5bmFIiwKXQoKCmRlZiBzaGEyNTZfZmlsZShwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBoYW5kbGU6CiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBoYW5kbGUucmVhZCgxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCgoKZGVmIGxvYWRfZmV0Y2hlcihyZXBvX3Jvb3Q6IFBhdGgpOgogICAgdGh1ID0gcmVwb19yb290IC8gIlRIVS1CREMyMDI2LW1haW4iCiAgICBtb2R1bGVfcGF0aCA9IHRodSAvICJnZXRfc3RvY2tfZGF0YS5weSIKICAgIGlmIG5vdCBtb2R1bGVfcGF0aC5pc19maWxlKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJtaXNzaW5nIHJlcG9zaXRvcnkgZmV0Y2hlcjoge21vZHVsZV9wYXRofSIpCiAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKHRodSkpCiAgICBpbXBvcnQgZ2V0X3N0b2NrX2RhdGEgYXMgZmV0Y2hlcgoKICAgIHJldHVybiBmZXRjaGVyCgoKZGVmIHRyYWRpbmdfZGF5c19iZXR3ZWVuKHN0YXJ0X2RhdGU6IHN0ciwgZW5kX2RhdGU6IHN0cikgLT4gbGlzdFtzdHJdOgogICAgaW1wb3J0IGJhb3N0b2NrIGFzIGJzCgogICAgcmVzdWx0ID0gYnMucXVlcnlfdHJhZGVfZGF0ZXMoc3RhcnRfZGF0ZT1zdGFydF9kYXRlLCBlbmRfZGF0ZT1lbmRfZGF0ZSkKICAgIGlmIHJlc3VsdC5lcnJvcl9jb2RlICE9ICIwIjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJxdWVyeV90cmFkZV9kYXRlcyBmYWlsZWQ6IHtyZXN1bHQuZXJyb3JfY29kZX0ge3Jlc3VsdC5lcnJvcl9tc2d9IikKICAgIGRheXMgPSBbXQogICAgd2hpbGUgcmVzdWx0LmVycm9yX2NvZGUgPT0gIjAiIGFuZCByZXN1bHQubmV4dCgpOgogICAgICAgIGNhbGVuZGFyX2RhdGUsIGlzX3RyYWRpbmcgPSByZXN1bHQuZ2V0X3Jvd19kYXRhKCkKICAgICAgICBpZiBzdHIoaXNfdHJhZGluZykgPT0gIjEiOgogICAgICAgICAgICBkYXlzLmFwcGVuZChjYWxlbmRhcl9kYXRlKQogICAgaWYgbm90IGRheXM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJCYW9TdG9jayByZXR1cm5lZCBubyB0cmFkaW5nIGRheXMiKQogICAgcmV0dXJuIGRheXMKCgpkZWYgY2FsZW5kYXJfY2FwKG5vdzogZGF0ZXRpbWUsIHRyYWRpbmdfZGF5czogbGlzdFtzdHJdKSAtPiBzdHI6CiAgICAiIiJMYXRlc3Qgc2Vzc2lvbiB0aGF0IGNhbiBiZSBjb21wbGV0ZSBhdCBgbm93YC4KCiAgICBgbm93YCBtdXN0IGJlIHRpbWV6b25lLWF3YXJlLiBBIHRyYWRpbmcgZGF5IGlzIGVsaWdpYmxlIG9ubHkgYXQgb3IgYWZ0ZXIKICAgIHRoZSAxNTowMCBBc2lhL1NoYW5naGFpIGNhc2ggY2xvc2UuIFRoZSBmdW5jdGlvbiBkb2VzIG5vdCBpbnZlbnQgYSBiYXIKICAgIHdoZW4gdGhlIHZlbmRvciBoYXMgbm90IHB1Ymxpc2hlZCBvbmU7IGNhbGxlcnMgc3RpbGwgaGF2ZSB0byByZWFkIHRoZQogICAgcmV0dXJuZWQgZGF0YS4KICAgICIiIgogICAgaWYgbm93LnR6aW5mbyBpcyBOb25lOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vdyBtdXN0IGJlIHRpbWV6b25lLWF3YXJlIikKICAgIGxvY2FsID0gbm93LmFzdGltZXpvbmUoU0hBTkdIQUkpCiAgICB0b2RheSA9IGxvY2FsLmRhdGUoKS5pc29mb3JtYXQoKQogICAgZWxhcHNlZCA9IFtkYXkgZm9yIGRheSBpbiB0cmFkaW5nX2RheXMgaWYgZGF5IDwgdG9kYXldCiAgICBpZiB0b2RheSBpbiB0cmFkaW5nX2RheXMgYW5kIGxvY2FsLnRpbWUoKSA+PSBDTE9TRV9USU1FOgogICAgICAgIHJldHVybiB0b2RheQogICAgaWYgbm90IGVsYXBzZWQ6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJubyBjb21wbGV0ZWQgdHJhZGluZyBkYXkgYmVmb3JlIHRoZSBjdXRvZmYiKQogICAgcmV0dXJuIGVsYXBzZWRbLTFdCgoKZGVmIGZldGNoX3VuaXZlcnNlKGZldGNoZXIsIHN0YXJ0X2RhdGU6IHN0ciwgZW5kX2RhdGU6IHN0ciwgcmVsb2dpbl9ldmVyeTogaW50ID0gMzApIC0+IHR1cGxlW3BkLkRhdGFGcmFtZSwgcGQuRGF0YUZyYW1lLCBsaXN0W2RpY3RdXToKICAgIGZldGNoZXIubG9naW4oKQogICAgY29uc3RpdHVlbnRzID0gZmV0Y2hlci5nZXRfaHMzMDBfc3RvY2tzKCkKICAgIGZyYW1lcyA9IFtdCiAgICBmYWlsdXJlcyA9IFtdCiAgICB0b3RhbCA9IGxlbihjb25zdGl0dWVudHMpCiAgICBmb3IgaW5kZXgsIHJvdyBpbiBjb25zdGl0dWVudHMuaXRlcnJvd3MoKToKICAgICAgICBic19jb2RlID0gcm93LmdldCgiY29kZSIsICIiKQogICAgICAgIHN0b2NrX25hbWUgPSByb3cuZ2V0KCJjb2RlX25hbWUiLCAiIikKICAgICAgICBhdHRlbXB0ID0gMAogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIGF0dGVtcHQgKz0gMQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBwcmludChmIlt7aW5kZXggKyAxfS97dG90YWx9XSB7YnNfY29kZX0ge3N0b2NrX25hbWV9IHtzdGFydF9kYXRlfS4ue2VuZF9kYXRlfSIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICBmcmFtZSA9IGZldGNoZXIuZ2V0X3N0b2NrX2hpc3RvcnkoYnNfY29kZSwgc3RhcnRfZGF0ZSwgZW5kX2RhdGUpCiAgICAgICAgICAgICAgICBpZiBmcmFtZSBpcyBOb25lIG9yIGZyYW1lLmVtcHR5OgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiZW1wdHkgaGlzdG9yeSIpCiAgICAgICAgICAgICAgICBmcmFtZSA9IGZldGNoZXIubm9ybWFsaXplX3N0b2NrX2NvZGVzKGZyYW1lKQogICAgICAgICAgICAgICAgZnJhbWVzLmFwcGVuZChmcmFtZSkKICAgICAgICAgICAgICAgIHByaW50KGYiICByb3dzPXtsZW4oZnJhbWUpfSBtYXg9e2ZyYW1lWyfml6XmnJ8nXS5tYXgoKX0iLCBmbHVzaD1UcnVlKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICBwcmludChmIiAgYXR0ZW1wdCB7YXR0ZW1wdH0gZmFpbGVkOiB7ZXhjfSIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICBpZiBhdHRlbXB0ID49IDM6CiAgICAgICAgICAgICAgICAgICAgZmFpbHVyZXMuYXBwZW5kKAogICAgICAgICAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiY29kZSI6IHN0cihic19jb2RlKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJuYW1lIjogc3RyKHN0b2NrX25hbWUpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgImVycm9yIjogc3RyKGV4YyksCiAgICAgICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGZldGNoZXIuZW5zdXJlX2xvZ2luKGZvcmNlPVRydWUpCiAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDEpCiAgICAgICAgaWYgZnJhbWVzIGFuZCBsZW4oZnJhbWVzKSAlIHJlbG9naW5fZXZlcnkgPT0gMDoKICAgICAgICAgICAgZmV0Y2hlci5lbnN1cmVfbG9naW4oZm9yY2U9VHJ1ZSkKICAgIGlmIG5vdCBmcmFtZXM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJubyBzdG9jayBoaXN0b3JpZXMgd2VyZSBkb3dubG9hZGVkIikKICAgIHBhbmVsID0gcGQuY29uY2F0KGZyYW1lcywgaWdub3JlX2luZGV4PVRydWUpCiAgICBwYW5lbCA9IGZldGNoZXIubm9ybWFsaXplX3N0b2NrX2NvZGVzKHBhbmVsKQogICAgcGFuZWwgPSBwYW5lbC5zb3J0X3ZhbHVlcyhbIuiCoeelqOS7o+eggSIsICLml6XmnJ8iXSkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgcmV0dXJuIGNvbnN0aXR1ZW50cywgcGFuZWwsIGZhaWx1cmVzCgoKZGVmIGF1ZGl0X3BhbmVsKHBhbmVsOiBwZC5EYXRhRnJhbWUsIGNvbnN0aXR1ZW50czogcGQuRGF0YUZyYW1lLCBlbmRfZGF0ZTogc3RyKSAtPiBkaWN0OgogICAgbWlzc2luZ19jb2x1bW5zID0gW2NvbHVtbiBmb3IgY29sdW1uIGluIFJFUVVJUkVEX0NPTFVNTlMgaWYgY29sdW1uIG5vdCBpbiBwYW5lbC5jb2x1bW5zXQogICAgaWYgbWlzc2luZ19jb2x1bW5zOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmImRvd25sb2FkZWQgcGFuZWwgbWlzc2luZyBjb2x1bW5zOiB7bWlzc2luZ19jb2x1bW5zfSIpCiAgICBjb2RlcyA9IHBhbmVsWyLogqHnpajku6PnoIEiXS5hc3R5cGUoc3RyKQogICAgZGF0ZXMgPSBwZC50b19kYXRldGltZShwYW5lbFsi5pel5pyfIl0sIGVycm9ycz0iY29lcmNlIikKICAgIGlmIGRhdGVzLmlzbmEoKS5hbnkoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ1bnBhcnNlYWJsZSBkYXRlczoge2ludChkYXRlcy5pc25hKCkuc3VtKCkpfSIpCiAgICBkdXBsaWNhdGVfcm93cyA9IGludChwYW5lbC5kdXBsaWNhdGVkKFsi6IKh56Wo5Luj56CBIiwgIuaXpeacnyJdKS5zdW0oKSkKICAgIGxhdGVyID0gc29ydGVkKHtkYXkgZm9yIGRheSBpbiBkYXRlcy5kdC5zdHJmdGltZSgiJVktJW0tJWQiKSBpZiBkYXkgPiBlbmRfZGF0ZX0pCiAgICBtYXhfZGF0ZSA9IGRhdGVzLm1heCgpLnN0cmZ0aW1lKCIlWS0lbS0lZCIpCiAgICBtaW5fZGF0ZSA9IGRhdGVzLm1pbigpLnN0cmZ0aW1lKCIlWS0lbS0lZCIpCiAgICBvbl9tYXggPSBwYW5lbC5sb2NbZGF0ZXMgPT0gZGF0ZXMubWF4KCksICLogqHnpajku6PnoIEiXS5hc3R5cGUoc3RyKQogICAgY29uc3RpdHVlbnRfY29kZXMgPSAoCiAgICAgICAgY29uc3RpdHVlbnRzWyJjb2RlIl0uYXN0eXBlKHN0cikuc3RyLnJlcGxhY2UoInNoLiIsICIiLCByZWdleD1GYWxzZSkuc3RyLnJlcGxhY2UoInN6LiIsICIiLCByZWdleD1GYWxzZSkuc3RyLnpmaWxsKDYpCiAgICApCiAgICBtaXNzaW5nX29uX21heCA9IHNvcnRlZChzZXQoY29uc3RpdHVlbnRfY29kZXMpIC0gc2V0KG9uX21heCkpCiAgICBiYWRfY29kZXMgPSBzb3J0ZWQoe2NvZGUgZm9yIGNvZGUgaW4gY29kZXMudW5pcXVlKCkgaWYgbGVuKGNvZGUpICE9IDYgb3Igbm90IGNvZGUuaXNkaWdpdCgpfSkKICAgIG51bGxfY291bnRzID0ge2NvbHVtbjogaW50KHBhbmVsW2NvbHVtbl0uaXNuYSgpLnN1bSgpKSBmb3IgY29sdW1uIGluIFJFUVVJUkVEX0NPTFVNTlN9CiAgICBvbl9tYXhfbWFzayA9IGRhdGVzID09IGRhdGVzLm1heCgpCiAgICBudWxsX3ZvbHVtZV9vbl9tYXggPSBpbnQoKG9uX21heF9tYXNrICYgcGFuZWxbIuaIkOS6pOmHjyJdLmlzbmEoKSkuc3VtKCkpCiAgICBwZXJfY29kZV9tYXggPSBwYW5lbC5hc3NpZ24oX2RhdGU9ZGF0ZXMpLmdyb3VwYnkoIuiCoeelqOS7o+eggSIpWyJfZGF0ZSJdLm1heCgpCiAgICByZXR1cm4gewogICAgICAgICJyb3dzIjogaW50KGxlbihwYW5lbCkpLAogICAgICAgICJzdG9ja3MiOiBpbnQoY29kZXMubnVuaXF1ZSgpKSwKICAgICAgICAiY29uc3RpdHVlbnRzIjogaW50KGNvbnN0aXR1ZW50X2NvZGVzLm51bmlxdWUoKSksCiAgICAgICAgIm1pbl9kYXRlIjogbWluX2RhdGUsCiAgICAgICAgIm1heF9kYXRlIjogbWF4X2RhdGUsCiAgICAgICAgInJlcXVlc3RlZF9lbmRfZGF0ZSI6IGVuZF9kYXRlLAogICAgICAgICJyb3dzX29uX21heF9kYXRlIjogaW50KGxlbihvbl9tYXgpKSwKICAgICAgICAic3RvY2tzX21pc3NpbmdfbWF4X2RhdGUiOiBtaXNzaW5nX29uX21heCwKICAgICAgICAiZGF0ZXNfYWZ0ZXJfcmVxdWVzdGVkX2VuZCI6IGxhdGVyLAogICAgICAgICJkdXBsaWNhdGVfY29kZV9kYXRlX3Jvd3MiOiBkdXBsaWNhdGVfcm93cywKICAgICAgICAiYmFkX3N0b2NrX2NvZGVzIjogYmFkX2NvZGVzLAogICAgICAgICJudWxsX2NvdW50cyI6IG51bGxfY291bnRzLAogICAgICAgICJudWxsX3ZvbHVtZV9yb3dzX29uX21heF9kYXRlIjogbnVsbF92b2x1bWVfb25fbWF4LAogICAgICAgICJ0cmFkaW5nX2RheXMiOiBpbnQoZGF0ZXMuZHQuc3RyZnRpbWUoIiVZLSVtLSVkIikubnVuaXF1ZSgpKSwKICAgICAgICAiZWFybGllc3Rfc3RvY2tfbWF4IjogcGVyX2NvZGVfbWF4Lm1pbigpLnN0cmZ0aW1lKCIlWS0lbS0lZCIpLAogICAgfQoKCmRlZiBjb21wbGV0ZW5lc3NfZXJyb3JzKGF1ZGl0OiBkaWN0LCBmYWlsdXJlczogbGlzdFtkaWN0XSkgLT4gbGlzdFtzdHJdOgogICAgZXJyb3JzID0gW10KICAgIGlmIGZhaWx1cmVzOgogICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7bGVuKGZhaWx1cmVzKX0gY29uc3RpdHVlbnRzIGZhaWxlZCB0byBkb3dubG9hZCIpCiAgICBpZiBhdWRpdFsiY29uc3RpdHVlbnRzIl0gIT0gMzAwOgogICAgICAgIGVycm9ycy5hcHBlbmQoZiJjb25zdGl0dWVudCBjb3VudCBpcyB7YXVkaXRbJ2NvbnN0aXR1ZW50cyddfSwgZXhwZWN0ZWQgMzAwIikKICAgIGlmIGF1ZGl0WyJzdG9ja3MiXSAhPSBhdWRpdFsiY29uc3RpdHVlbnRzIl06CiAgICAgICAgZXJyb3JzLmFwcGVuZCgiZG93bmxvYWRlZCBzdG9jayBjb3VudCBkb2VzIG5vdCBtYXRjaCB0aGUgY29uc3RpdHVlbnQgbGlzdCIpCiAgICBpZiBhdWRpdFsiZGF0ZXNfYWZ0ZXJfcmVxdWVzdGVkX2VuZCJdOgogICAgICAgIGVycm9ycy5hcHBlbmQoZiJwYW5lbCBjb250YWlucyBkYXRlcyBhZnRlciB0aGUgY3V0b2ZmOiB7YXVkaXRbJ2RhdGVzX2FmdGVyX3JlcXVlc3RlZF9lbmQnXX0iKQogICAgaWYgYXVkaXRbIm1heF9kYXRlIl0gIT0gYXVkaXRbInJlcXVlc3RlZF9lbmRfZGF0ZSJdOgogICAgICAgIGVycm9ycy5hcHBlbmQoCiAgICAgICAgICAgIGYiYWN0dWFsIG1heCBkYXRlIHthdWRpdFsnbWF4X2RhdGUnXX0gaXMgZWFybGllciB0aGFuIHRoZSByZXF1ZXN0ZWQgY29tcGxldGUgc2Vzc2lvbiB7YXVkaXRbJ3JlcXVlc3RlZF9lbmRfZGF0ZSddfSIKICAgICAgICApCiAgICBpZiBhdWRpdFsiZHVwbGljYXRlX2NvZGVfZGF0ZV9yb3dzIl06CiAgICAgICAgZXJyb3JzLmFwcGVuZCgiZHVwbGljYXRlIHN0b2NrL2RhdGUgcm93cyIpCiAgICBpZiBhdWRpdFsiYmFkX3N0b2NrX2NvZGVzIl06CiAgICAgICAgZXJyb3JzLmFwcGVuZChmInN0b2NrIGNvZGVzIGFyZSBub3QgNiBkaWdpdHM6IHthdWRpdFsnYmFkX3N0b2NrX2NvZGVzJ11bOjEwXX0iKQogICAgIyBBIGZ1bGwgQ1NJIDMwMCBjbG9zZSBzaG91bGQgYmUgcHJlc2VudCBmb3IgYWxtb3N0IGV2ZXJ5IG5hbWUuIEEgc21hbGwKICAgICMgbWlzcyBjYW4gYmUgYSByZWFsIHN1c3BlbnNpb247IGEgbGFyZ2UgbWlzcyBtZWFucyB0aGUgc25hcHNob3QgaXMgbm90IGNvbXBsZXRlLgogICAgaWYgYXVkaXRbInJvd3Nfb25fbWF4X2RhdGUiXSA8IDI5MDoKICAgICAgICBlcnJvcnMuYXBwZW5kKAogICAgICAgICAgICBmIm9ubHkge2F1ZGl0Wydyb3dzX29uX21heF9kYXRlJ119IHN0b2NrcyBoYXZlIGEgYmFyIG9uIHthdWRpdFsnbWF4X2RhdGUnXX07IHJlZnVzaW5nIHRvIGNhbGwgdGhlIHBhbmVsIGNvbXBsZXRlIgogICAgICAgICkKICAgIGZvciBjb2x1bW4gaW4gKCLlvIDnm5giLCAi5pS255uYIiwgIuacgOmrmCIsICLmnIDkvY4iKToKICAgICAgICBpZiBhdWRpdFsibnVsbF9jb3VudHMiXS5nZXQoY29sdW1uLCAwKToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm51bGwgcHJpY2VzIGluIHtjb2x1bW59IikKICAgICMgSGFsdGVkIHNlc3Npb25zIG9mdGVuIGhhdmUgYSBwcmljZSBhbmQgYW4gZW1wdHkgdm9sdW1lLiBUaGV5IGFyZSBub3QgYW4KICAgICMgdW5maW5pc2hlZCBtYXJrZXQgZGF5LiBBIG1pc3Npbmcgdm9sdW1lIG9uIHRoZSByZXF1ZXN0ZWQgZmluYWwgc2Vzc2lvbiBpcy4KICAgIGlmIGF1ZGl0LmdldCgibnVsbF92b2x1bWVfcm93c19vbl9tYXhfZGF0ZSIsIDApOgogICAgICAgIGVycm9ycy5hcHBlbmQoInRoZSBmaW5hbCBzZXNzaW9uIGhhcyByb3dzIHdpdGggZW1wdHkgdm9sdW1lIikKICAgIHJldHVybiBlcnJvcnMKCgpkZWYgd3JpdGVfb3V0cHV0cyhvdXRwdXRfZGlyOiBQYXRoLCBjb25zdGl0dWVudHM6IHBkLkRhdGFGcmFtZSwgcGFuZWw6IHBkLkRhdGFGcmFtZSwgYXVkaXQ6IGRpY3QsIGZhaWx1cmVzOiBsaXN0W2RpY3RdLCBtZXRhOiBkaWN0KSAtPiBkaWN0OgogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwYW5lbF9wYXRoID0gb3V0cHV0X2RpciAvICJoczMwMF9kYWlseV9oZnFfMjAyNDAxMDFfY29tcGxldGUuY3N2IgogICAgbGlzdF9wYXRoID0gb3V0cHV0X2RpciAvICJoczMwMF9jb25zdGl0dWVudHMuY3N2IgogICAgbWFuaWZlc3RfcGF0aCA9IG91dHB1dF9kaXIgLyAiaHMzMDBfZGF0YV9tYW5pZmVzdC5qc29uIgogICAgcGFuZWwudG9fY3N2KHBhbmVsX3BhdGgsIGluZGV4PUZhbHNlLCBlbmNvZGluZz0idXRmLTgtc2lnIikKICAgIGNvbnN0aXR1ZW50cy50b19jc3YobGlzdF9wYXRoLCBpbmRleD1GYWxzZSwgZW5jb2Rpbmc9InV0Zi04LXNpZyIpCiAgICBtYW5pZmVzdCA9IHsKICAgICAgICAqKm1ldGEsCiAgICAgICAgImF1ZGl0IjogYXVkaXQsCiAgICAgICAgImZhaWx1cmVzIjogZmFpbHVyZXMsCiAgICAgICAgImZpbGVzIjogewogICAgICAgICAgICAicGFuZWwiOiB7CiAgICAgICAgICAgICAgICAicGF0aCI6IHN0cihwYW5lbF9wYXRoKSwKICAgICAgICAgICAgICAgICJzaGEyNTYiOiBzaGEyNTZfZmlsZShwYW5lbF9wYXRoKSwKICAgICAgICAgICAgICAgICJieXRlcyI6IHBhbmVsX3BhdGguc3RhdCgpLnN0X3NpemUsCiAgICAgICAgICAgIH0sCiAgICAgICAgICAgICJjb25zdGl0dWVudHMiOiB7CiAgICAgICAgICAgICAgICAicGF0aCI6IHN0cihsaXN0X3BhdGgpLAogICAgICAgICAgICAgICAgInNoYTI1NiI6IHNoYTI1Nl9maWxlKGxpc3RfcGF0aCksCiAgICAgICAgICAgICAgICAiYnl0ZXMiOiBsaXN0X3BhdGguc3RhdCgpLnN0X3NpemUsCiAgICAgICAgICAgIH0sCiAgICAgICAgfSwKICAgIH0KICAgIG1hbmlmZXN0X3BhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKG1hbmlmZXN0LCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIG1hbmlmZXN0WyJmaWxlcyJdWyJtYW5pZmVzdCJdID0gc3RyKG1hbmlmZXN0X3BhdGgpCiAgICByZXR1cm4gbWFuaWZlc3QKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJEb3dubG9hZCB0aGUgbGF0ZXN0IGNvbXBsZXRlIENTSSAzMDAgZGFpbHkgcGFuZWwgdmlhIEJhb1N0b2NrIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVwby1yb290IiwgdHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoIi93b3Jrc3BhY2UiKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCIvb3B0L2N1cnNvci9hcnRpZmFjdHMiKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc3RhcnQtZGF0ZSIsIGRlZmF1bHQ9REVGQVVMVF9TVEFSVCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbm93IiwgZGVmYXVsdD0iIiwgaGVscD0iT3ZlcnJpZGUgQXNpYS9TaGFuZ2hhaSBub3csIElTTyBmb3JtYXQsIGZvciB0ZXN0cyBvbmx5IikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgc29ja2V0LnNldGRlZmF1bHR0aW1lb3V0KDkwKQogICAgZmV0Y2hlciA9IGxvYWRfZmV0Y2hlcihhcmdzLnJlcG9fcm9vdCkKICAgIG5vdyA9IGRhdGV0aW1lLmZyb21pc29mb3JtYXQoYXJncy5ub3cpIGlmIGFyZ3Mubm93IGVsc2UgZGF0ZXRpbWUubm93KFNIQU5HSEFJKQogICAgaWYgbm93LnR6aW5mbyBpcyBOb25lOgogICAgICAgIG5vdyA9IG5vdy5yZXBsYWNlKHR6aW5mbz1TSEFOR0hBSSkKCiAgICBmZXRjaGVyLmxvZ2luKCkKICAgIHRyeToKICAgICAgICB0cmFkaW5nX2RheXMgPSB0cmFkaW5nX2RheXNfYmV0d2VlbigiMjAyNC0wMS0wMSIsICIyMDI3LTA2LTMwIikKICAgICAgICBlbmRfZGF0ZSA9IGNhbGVuZGFyX2NhcChub3csIHRyYWRpbmdfZGF5cykKICAgICAgICBwcmludChmIkFzaWEvU2hhbmdoYWkgbm93OiB7bm93Lmlzb2Zvcm1hdCgpfSIsIGZsdXNoPVRydWUpCiAgICAgICAgcHJpbnQoZiJDb21wbGV0ZS1zZXNzaW9uIGNhcDoge2VuZF9kYXRlfSIsIGZsdXNoPVRydWUpCiAgICAgICAgY29uc3RpdHVlbnRzLCBwYW5lbCwgZmFpbHVyZXMgPSBmZXRjaF91bml2ZXJzZShmZXRjaGVyLCBhcmdzLnN0YXJ0X2RhdGUsIGVuZF9kYXRlKQogICAgICAgIGF1ZGl0ID0gYXVkaXRfcGFuZWwocGFuZWwsIGNvbnN0aXR1ZW50cywgZW5kX2RhdGUpCiAgICAgICAgZXJyb3JzID0gY29tcGxldGVuZXNzX2Vycm9ycyhhdWRpdCwgZmFpbHVyZXMpCiAgICAgICAgbWV0YSA9IHsKICAgICAgICAgICAgInNvdXJjZSI6ICJiYW9zdG9jayIsCiAgICAgICAgICAgICJlbmRwb2ludCI6ICJwdWJsaWMtYXBpLmJhb3N0b2NrLmNvbToxMDAzMCIsCiAgICAgICAgICAgICJhZGp1c3RmbGFnIjogIjEiLAogICAgICAgICAgICAiYWRqdXN0ZmxhZ19tZWFuaW5nIjogIuWQjuWkjeadgyIsCiAgICAgICAgICAgICJmaWVsZHMiOiAiZGF0ZSxjb2RlLG9wZW4saGlnaCxsb3csY2xvc2UscHJlY2xvc2Usdm9sdW1lLGFtb3VudCx0dXJuLHBjdENoZyIsCiAgICAgICAgICAgICJyZXBvc2l0b3J5X2ZldGNoZXIiOiAiVEhVLUJEQzIwMjYtbWFpbi9nZXRfc3RvY2tfZGF0YS5weTpnZXRfc3RvY2tfaGlzdG9yeSIsCiAgICAgICAgICAgICJzdGFydF9kYXRlIjogYXJncy5zdGFydF9kYXRlLAogICAgICAgICAgICAiYXNpYV9zaGFuZ2hhaV9ub3ciOiBub3cuaXNvZm9ybWF0KCksCiAgICAgICAgICAgICJjb21wbGV0ZV9zZXNzaW9uX2NhcCI6IGVuZF9kYXRlLAogICAgICAgICAgICAiY29tcGxldGUiOiBub3QgZXJyb3JzLAogICAgICAgICAgICAiY29tcGxldGVuZXNzX2Vycm9ycyI6IGVycm9ycywKICAgICAgICAgICAgInBpbm5lZF9jb21taXRfZXhwZWN0ZWQiOiAiYzczNTc5N2M4NmVjMzkwODM3OWY3ZjU0YjcwOTUzYzZhNjJiMjNiOSIsCiAgICAgICAgICAgICJ0cmFkaW5nX2RheXMiOiB0cmFkaW5nX2RheXMsCiAgICAgICAgfQogICAgICAgIG1hbmlmZXN0ID0gd3JpdGVfb3V0cHV0cyhhcmdzLm91dHB1dF9kaXIsIGNvbnN0aXR1ZW50cywgcGFuZWwsIGF1ZGl0LCBmYWlsdXJlcywgbWV0YSkKICAgIGZpbmFsbHk6CiAgICAgICAgZmV0Y2hlci5sb2dvdXQoKQoKICAgIHByaW50KGpzb24uZHVtcHMoeyJjb21wbGV0ZSI6IG1hbmlmZXN0WyJjb21wbGV0ZSJdLCAiYXVkaXQiOiBtYW5pZmVzdFsiYXVkaXQiXSwgImVycm9ycyI6IG1hbmlmZXN0WyJjb21wbGV0ZW5lc3NfZXJyb3JzIl19LCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSkKICAgIHJldHVybiAwIGlmIG1hbmlmZXN0WyJjb21wbGV0ZSJdIGVsc2UgMgoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB0cnk6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHRyYWNlYmFjay5wcmludF9leGMoKQogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoMSkK", "export_ranking.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJFeHBvcnQgdGhlIGZ1bGwgY3Jvc3Mtc2VjdGlvbiByYW5raW5nIGFuZCB0aGUgb2ZmaWNpYWwgdG9wLTUgZmlsZS4KClRoaXMgaXMgdGhlIGluZmVyZW5jZSBwYXRoIGZvciB0aGUgS2FnZ2xlIHJ1bi4gRmVhdHVyZSBjb25zdHJ1Y3Rpb24sIHRoZQpsZWFkZXIgcG9vbCwgdGhlIHZvbGF0aWxpdHkgZmlsdGVyLCB0aGUgZW5zZW1ibGUgbWVhbiwgYW5kIHRoZSB0b3AtNSB3ZWlnaHRzCmFyZSB0aGUgb25lcyBpbXBsZW1lbnRlZCBpbiBhcHAvY29kZS9wcmVkaWN0LnB5IGF0IHRoZSBwaW5uZWQgY29tbWl0LiBUaGUKZXh0cmEgZmlsZSBhZGRzIHRoZSBzdG9jayBuYW1lLCB0aGUgbGFiZWwgaG9yaXpvbiwgYW5kIGV2ZXJ5IHNjb3JlZCBuYW1lLgpJdCBkb2VzIG5vdCBmaXQgYSBuZXcgbW9kZWwgYW5kIGl0IGRvZXMgbm90IGNoYW5nZSBjb25maWcucHkuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCkhPUklaT04gPSAi5pyq5p2lNeS4quS6pOaYk+aXpeW8gOebmOWIsOW8gOebmOaUtuebijogKG9wZW5bdCs1XS1vcGVuW3QrMV0pL29wZW5bdCsxXSIKTEFCRUxfRk9STVVMQSA9ICIob3Blblt0KzVdLW9wZW5bdCsxXSkvb3Blblt0KzFdIgoKCmRlZiBfbG9hZF9uYW1lcyhwYXRoOiBQYXRoKSAtPiBkaWN0W3N0ciwgc3RyXToKICAgIGZyYW1lID0gcGQucmVhZF9jc3YocGF0aCwgZHR5cGU9c3RyLCBlbmNvZGluZz0idXRmLTgtc2lnIikKICAgIGNvZGVfY29sdW1uID0gImNvZGUiIGlmICJjb2RlIiBpbiBmcmFtZS5jb2x1bW5zIGVsc2UgIuiCoeelqOS7o+eggSIKICAgIG5hbWVfY29sdW1uID0gImNvZGVfbmFtZSIgaWYgImNvZGVfbmFtZSIgaW4gZnJhbWUuY29sdW1ucyBlbHNlICLogqHnpajlkI3np7AiCiAgICBuYW1lcyA9IHt9CiAgICBmb3IgXywgcm93IGluIGZyYW1lLml0ZXJyb3dzKCk6CiAgICAgICAgY29kZSA9IHN0cihyb3dbY29kZV9jb2x1bW5dKS5yZXBsYWNlKCJzaC4iLCAiIikucmVwbGFjZSgic3ouIiwgIiIpLnJlcGxhY2UoIi4wIiwgIiIpCiAgICAgICAgY29kZSA9IGNvZGUuemZpbGwoNikKICAgICAgICBuYW1lc1tjb2RlXSA9ICIiIGlmIHBkLmlzbmEocm93W25hbWVfY29sdW1uXSkgZWxzZSBzdHIocm93W25hbWVfY29sdW1uXSkKICAgIHJldHVybiBuYW1lcwoKCmRlZiBfbnRoX3RyYWRpbmdfZGF5KHRyYWRpbmdfZGF5czogbGlzdFtzdHJdLCBhc29mOiBzdHIsIHN0ZXA6IGludCkgLT4gc3RyOgogICAgaWYgYXNvZiBub3QgaW4gdHJhZGluZ19kYXlzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7YXNvZn0gaXMgbm90IGluIHRoZSB0cmFkaW5nIGNhbGVuZGFyIikKICAgIGluZGV4ID0gdHJhZGluZ19kYXlzLmluZGV4KGFzb2YpICsgc3RlcAogICAgaWYgaW5kZXggPCAwIG9yIGluZGV4ID49IGxlbih0cmFkaW5nX2RheXMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJjYW5ub3QgcmVzb2x2ZSB0K3tzdGVwfSBmcm9tIHthc29mfSIpCiAgICByZXR1cm4gdHJhZGluZ19kYXlzW2luZGV4XQoKCmRlZiBleHBvcnRfcmFua2luZyhuYW1lc19jc3Y6IFBhdGgsIG91dF9jc3Y6IFBhdGgsIHByb3ZlbmFuY2VfcGF0aDogUGF0aCkgLT4gcGQuRGF0YUZyYW1lOgogICAgaW1wb3J0IG11bHRpcHJvY2Vzc2luZyBhcyBtcAoKICAgIGltcG9ydCBqb2JsaWIKICAgIGltcG9ydCB0b3JjaAoKICAgIG1wLnNldF9zdGFydF9tZXRob2QoInNwYXduIiwgZm9yY2U9VHJ1ZSkKCiAgICBmcm9tIGNvbmZpZyBpbXBvcnQgY29uZmlnCiAgICBmcm9tIG1vZGVsIGltcG9ydCBTdG9ja1RyYW5zZm9ybWVyCiAgICBmcm9tIHByZWRpY3QgaW1wb3J0ICgKICAgICAgICBfZmluZF9tb2RlbF9yb290LAogICAgICAgIF9maW5kX3NlZWRfZGlyLAogICAgICAgIGJ1aWxkX2luZmVyZW5jZV9zZXF1ZW5jZXMsCiAgICAgICAgcHJlcHJvY2Vzc19wcmVkaWN0X2RhdGEsCiAgICApCgogICAgcHJvdmVuYW5jZSA9IGpzb24ubG9hZHMocHJvdmVuYW5jZV9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHJlcXVpcmVkX3NlZWRzID0gW2ludChzZWVkKSBmb3Igc2VlZCBpbiBwcm92ZW5hbmNlWyJyZXF1aXJlZF9zZWVkcyJdXQogICAgaWYgcHJvdmVuYW5jZS5nZXQoInJ1bl9tb2RlIikgPT0gImZ1bGwiIGFuZCByZXF1aXJlZF9zZWVkcyAhPSBbNDIsIDEyM106CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiZnVsbCByYW5raW5nIHJlcXVpcmVzIHNlZWRzIFs0MiwgMTIzXSwgZ290IHtyZXF1aXJlZF9zZWVkc30iKQoKICAgIGVuc2VtYmxlX3NlZWRzID0gY29uZmlnLmdldCgiZW5zZW1ibGVfc2VlZHMiLCBbNDJdKQogICAgZGF0YV9maWxlID0gb3MucGF0aC5qb2luKGNvbmZpZ1siZGF0YV9wYXRoIl0sIGNvbmZpZy5nZXQoInByZWRpY3RfZmlsZSIsICJ0ZXN0LmNzdiIpKQogICAgb3V0cHV0X2RpciA9IF9maW5kX21vZGVsX3Jvb3QoY29uZmlnWyJvdXRwdXRfZGlyIl0pCiAgICBtb2RlbF9jb25maWdfcGF0aCA9IG9zLnBhdGguam9pbihvdXRwdXRfZGlyLCAiY29uZmlnLmpzb24iKQogICAgaWYgb3MucGF0aC5leGlzdHMobW9kZWxfY29uZmlnX3BhdGgpOgogICAgICAgIHdpdGggb3Blbihtb2RlbF9jb25maWdfcGF0aCwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIG1vZGVsX2NvbmZpZyA9IGpzb24ubG9hZChoYW5kbGUpCiAgICAgICAgZm9yIGtleSBpbiBbCiAgICAgICAgICAgICJzZXF1ZW5jZV9sZW5ndGgiLAogICAgICAgICAgICAiZF9tb2RlbCIsCiAgICAgICAgICAgICJuaGVhZCIsCiAgICAgICAgICAgICJudW1fbGF5ZXJzIiwKICAgICAgICAgICAgImRpbV9mZWVkZm9yd2FyZCIsCiAgICAgICAgICAgICJkcm9wb3V0IiwKICAgICAgICAgICAgImZlYXR1cmVfbnVtIiwKICAgICAgICAgICAgImNyb3NzX3N0b2NrX2xheWVycyIsCiAgICAgICAgICAgICJ0b3Bfa19vdXRwdXQiLAogICAgICAgICAgICAidG9wX2tfd2VpZ2h0cyIsCiAgICAgICAgICAgICJwcmVkaWN0X3VzZV9sZWFkZXJfcG9vbCIsCiAgICAgICAgXToKICAgICAgICAgICAgaWYga2V5IGluIG1vZGVsX2NvbmZpZzoKICAgICAgICAgICAgICAgIGNvbmZpZ1trZXldID0gbW9kZWxfY29uZmlnW2tleV0KCiAgICBkZWYgcmVzb2x2ZShuYW1lOiBzdHIpIC0+IHN0cjoKICAgICAgICBwYXRoID0gb3MucGF0aC5qb2luKG91dHB1dF9kaXIsIG5hbWUpCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMocGF0aCk6CiAgICAgICAgICAgIHJldHVybiBwYXRoCiAgICAgICAgcmV0dXJuIG9zLnBhdGguam9pbihfZmluZF9zZWVkX2RpcihvdXRwdXRfZGlyLCBlbnNlbWJsZV9zZWVkc1swXSksIG5hbWUpCgogICAgc2NhbGVyX3BhdGggPSByZXNvbHZlKCJzY2FsZXIucGtsIikKICAgIHN0b2NraWQyaWR4X3BhdGggPSByZXNvbHZlKCJzdG9ja2lkMmlkeC5wa2wiKQogICAgbWVkaWFuc19wYXRoID0gcmVzb2x2ZSgidHJhaW5fbWVkaWFucy5wa2wiKQogICAgbGVhZGVyX3BhdGggPSByZXNvbHZlKCJsZWFkZXJfc3RvY2tfaWRzLnBrbCIpCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoc2NhbGVyX3BhdGgpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKHNjYWxlcl9wYXRoKQogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKGRhdGFfZmlsZSk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZGF0YV9maWxlKQoKICAgIHJhd19kZiA9IHBkLnJlYWRfY3N2KGRhdGFfZmlsZSwgZHR5cGU9eyLogqHnpajku6PnoIEiOiBzdHJ9KQogICAgcmF3X2RmWyLogqHnpajku6PnoIEiXSA9IHJhd19kZlsi6IKh56Wo5Luj56CBIl0uYXN0eXBlKHN0cikuc3RyLnJlcGxhY2UociJcLjAkIiwgIiIsIHJlZ2V4PVRydWUpLnN0ci56ZmlsbCg2KQogICAgcmF3X2RmWyLml6XmnJ8iXSA9IHBkLnRvX2RhdGV0aW1lKHJhd19kZlsi5pel5pyfIl0pCiAgICBsYXRlc3RfZGF0ZSA9IHJhd19kZlsi5pel5pyfIl0ubWF4KCkKICAgIGFzb2YgPSBsYXRlc3RfZGF0ZS5zdHJmdGltZSgiJVktJW0tJWQiKQogICAgaWYgcHJvdmVuYW5jZS5nZXQoImV4cGVjdGVkX2Fzb2ZfZGF0ZSIpIGFuZCBhc29mICE9IHByb3ZlbmFuY2VbImV4cGVjdGVkX2Fzb2ZfZGF0ZSJdOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmInByZWRpY3QgZmlsZSBtYXggZGF0ZSB7YXNvZn0gIT0gZXhwZWN0ZWQge3Byb3ZlbmFuY2VbJ2V4cGVjdGVkX2Fzb2ZfZGF0ZSddfSIpCgogICAgcmF3X21hcCA9IGpvYmxpYi5sb2FkKHN0b2NraWQyaWR4X3BhdGgpCiAgICBzdG9ja2lkMmlkeCA9IHtzdHIoc2lkKS5yZXBsYWNlKCIuMCIsICIiKS56ZmlsbCg2KTogaW50KGlkeCkgZm9yIHNpZCwgaWR4IGluIHJhd19tYXAuaXRlbXMoKX0KICAgIHN0b2NrX2lkcyA9IHNvcnRlZChzdG9ja2lkMmlkeC5rZXlzKCkpCiAgICBpZiBjb25maWcuZ2V0KCJwcmVkaWN0X3VzZV9sZWFkZXJfcG9vbCIsIFRydWUpIGFuZCBvcy5wYXRoLmV4aXN0cyhsZWFkZXJfcGF0aCk6CiAgICAgICAgbGVhZGVyX3N0b2NrX2lkcyA9IHtzdHIocykucmVwbGFjZSgiLjAiLCAiIikuemZpbGwoNikgZm9yIHMgaW4gam9ibGliLmxvYWQobGVhZGVyX3BhdGgpfQogICAgICAgIHN0b2NrX2lkcyA9IFtzaWQgZm9yIHNpZCBpbiBzdG9ja19pZHMgaWYgc2lkIGluIGxlYWRlcl9zdG9ja19pZHNdCiAgICAgICAgcHJpbnQoZiLpooTmtYvpmZDliLblnKjorq3nu4PpvpnlpLTmsaA6IHtsZW4oc3RvY2tfaWRzKX0g5Y+qIikKICAgIGVsaWYgY29uZmlnLmdldCgicHJlZGljdF91c2VfbGVhZGVyX3Bvb2wiLCBUcnVlKToKICAgICAgICBwcmludCgi6K2m5ZGKOiDmnKrmib7liLAgbGVhZGVyX3N0b2NrX2lkcy5wa2zvvIzlm57pgIDkuLrlhajluILlnLrpooTmtYsiKQoKICAgIHByb2Nlc3NlZCwgZmVhdHVyZXMgPSBwcmVwcm9jZXNzX3ByZWRpY3RfZGF0YShyYXdfZGYsIHN0b2NraWQyaWR4KQogICAgcHJvY2Vzc2VkW2ZlYXR1cmVzXSA9IHByb2Nlc3NlZFtmZWF0dXJlc10ucmVwbGFjZShbbnAuaW5mLCAtbnAuaW5mXSwgbnAubmFuKQogICAgaWYgb3MucGF0aC5leGlzdHMobWVkaWFuc19wYXRoKToKICAgICAgICBwcm9jZXNzZWRbZmVhdHVyZXNdID0gcHJvY2Vzc2VkW2ZlYXR1cmVzXS5maWxsbmEoam9ibGliLmxvYWQobWVkaWFuc19wYXRoKSkKICAgIGVsc2U6CiAgICAgICAgcHJvY2Vzc2VkW2ZlYXR1cmVzXSA9IHByb2Nlc3NlZFtmZWF0dXJlc10uZmlsbG5hKDAuMCkKCiAgICBzY2FsZXIgPSBqb2JsaWIubG9hZChzY2FsZXJfcGF0aCkKICAgIHByb2Nlc3NlZFtmZWF0dXJlc10gPSBzY2FsZXIudHJhbnNmb3JtKHByb2Nlc3NlZFtmZWF0dXJlc10pCiAgICBzZXF1ZW5jZXNfbnAsIHNlcXVlbmNlX3N0b2NrX2lkcyA9IGJ1aWxkX2luZmVyZW5jZV9zZXF1ZW5jZXMoCiAgICAgICAgcHJvY2Vzc2VkLAogICAgICAgIGZlYXR1cmVzLAogICAgICAgIGNvbmZpZ1sic2VxdWVuY2VfbGVuZ3RoIl0sCiAgICAgICAgc3RvY2tfaWRzLAogICAgICAgIGxhdGVzdF9kYXRlLAogICAgKQoKICAgIGV4Y2x1ZGVkID0gW10KICAgIGlmICJ2b2xhdGlsaXR5XzIwIiBpbiBmZWF0dXJlczoKICAgICAgICB2b2xfaWR4ID0gZmVhdHVyZXMuaW5kZXgoInZvbGF0aWxpdHlfMjAiKQogICAgICAgIGxhc3Rfdm9sID0gc2VxdWVuY2VzX25wWzosIC0xLCB2b2xfaWR4XQogICAgICAgIHZvbF9tZWFuID0gZmxvYXQobnAubmFubWVhbihsYXN0X3ZvbCkpCiAgICAgICAgdm9sX3N0ZCA9IGZsb2F0KG5wLm5hbnN0ZChsYXN0X3ZvbCkpCiAgICAgICAgdGhyZXNob2xkID0gdm9sX21lYW4gKyAyLjAgKiB2b2xfc3RkCiAgICAgICAga2VlcCA9IGxhc3Rfdm9sIDw9IHRocmVzaG9sZAogICAgICAgIG1pbl9rZWVwID0gbWF4KGNvbmZpZy5nZXQoInRvcF9rX291dHB1dCIsIDUpICogMywgMjApCiAgICAgICAgaWYgKG5vdCBucC5hbGwoa2VlcCkpIGFuZCBpbnQoa2VlcC5zdW0oKSkgPj0gbWluX2tlZXA6CiAgICAgICAgICAgIGV4Y2x1ZGVkID0gW3NpZCBmb3Igc2lkLCBmbGFnIGluIHppcChzZXF1ZW5jZV9zdG9ja19pZHMsIGtlZXApIGlmIG5vdCBmbGFnXQogICAgICAgICAgICBzZXF1ZW5jZXNfbnAgPSBzZXF1ZW5jZXNfbnBba2VlcF0KICAgICAgICAgICAgc2VxdWVuY2Vfc3RvY2tfaWRzID0gW3NpZCBmb3Igc2lkLCBmbGFnIGluIHppcChzZXF1ZW5jZV9zdG9ja19pZHMsIGtlZXApIGlmIGZsYWddCiAgICAgICAgICAgIHByaW50KGYi6aOO6Zmp6L+H5rukOiDmjpLpmaQge2xlbihleGNsdWRlZCl9IOWPqu+8jOS/neeVmSB7bGVuKHNlcXVlbmNlX3N0b2NrX2lkcyl9IOWPqiIpCgogICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiKQogICAgZWxpZiB0b3JjaC5iYWNrZW5kcy5tcHMuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJtcHMiKQogICAgZWxzZToKICAgICAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImNwdSIpCiAgICBpZiBwcm92ZW5hbmNlLmdldCgicnVuX21vZGUiKSA9PSAiZnVsbCIgYW5kIGRldmljZS50eXBlICE9ICJjdWRhIjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImZ1bGwgcmFua2luZyByZXF1aXJlcyBDVURBOyByZWZ1c2luZyBDUFUgc2NvcmVzIikKCiAgICBtb2RlbCA9IFN0b2NrVHJhbnNmb3JtZXIoaW5wdXRfZGltPWxlbihmZWF0dXJlcyksIGNvbmZpZz1jb25maWcsIG51bV9zdG9ja3M9bGVuKHN0b2NraWQyaWR4KSkKICAgIG1vZGVsLnRvKGRldmljZSkKICAgIG1vZGVsLmV2YWwoKQogICAgcGVyX3NlZWQgPSBbXQogICAgbG9hZGVkID0gW10KICAgIGZvciBzZWVkIGluIGVuc2VtYmxlX3NlZWRzOgogICAgICAgIG1vZGVsX3BhdGggPSBvcy5wYXRoLmpvaW4oX2ZpbmRfc2VlZF9kaXIob3V0cHV0X2Rpciwgc2VlZCksICJiZXN0X21vZGVsLnB0aCIpCiAgICAgICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKG1vZGVsX3BhdGgpOgogICAgICAgICAgICBwcmludChmIuitpuWRijogc2VlZD17c2VlZH0g5qih5Z6L5LiN5a2Y5Zyo77yM6Lez6L+HIikKICAgICAgICAgICAgY29udGludWUKICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QodG9yY2gubG9hZChtb2RlbF9wYXRoLCBtYXBfbG9jYXRpb249ZGV2aWNlKSkKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgc2NvcmVzID0gbW9kZWwodG9yY2guZnJvbV9udW1weShzZXF1ZW5jZXNfbnApLnVuc3F1ZWV6ZSgwKS50byhkZXZpY2UpKS5zcXVlZXplKDApLmRldGFjaCgpLmNwdSgpLm51bXB5KCkKICAgICAgICBwZXJfc2VlZC5hcHBlbmQoc2NvcmVzKQogICAgICAgIGxvYWRlZC5hcHBlbmQoaW50KHNlZWQpKQogICAgICAgIHByaW50KGYic2VlZD17c2VlZH0g5qih5Z6L5bey5Yqg6L295bm26aKE5rWLIikKICAgIGlmIGxvYWRlZCAhPSByZXF1aXJlZF9zZWVkczoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJsb2FkZWQgc2VlZHMge2xvYWRlZH0gIT0gcmVxdWlyZWQge3JlcXVpcmVkX3NlZWRzfSIpCgogICAgc2NvcmVzID0gbnAubWVhbihwZXJfc2VlZCwgYXhpcz0wKQogICAgb3JkZXIgPSBucC5hcmdzb3J0KHNjb3JlcylbOjotMV0KICAgIHRvcF9rID0gaW50KGNvbmZpZy5nZXQoInRvcF9rX291dHB1dCIsIDUpKQogICAgd2VpZ2h0cyA9IFtmbG9hdCh2YWx1ZSkgZm9yIHZhbHVlIGluIGNvbmZpZy5nZXQoInRvcF9rX3dlaWdodHMiKV0KICAgIGlmIGxlbih3ZWlnaHRzKSAhPSB0b3BfazoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ0b3Bfa193ZWlnaHRzIGxlbmd0aCB7bGVuKHdlaWdodHMpfSAhPSB0b3Bfa19vdXRwdXQge3RvcF9rfSIpCiAgICBpZiBwcm92ZW5hbmNlLmdldCgicnVuX21vZGUiKSA9PSAiZnVsbCIgYW5kIHdlaWdodHMgIT0gWzAuNTAsIDAuNDcsIDAuMDEsIDAuMDEsIDAuMDFdOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmInVuZXhwZWN0ZWQgd2VpZ2h0cyB7d2VpZ2h0c30iKQoKICAgIGxhc3RfYmFyID0gcmF3X2RmLmdyb3VwYnkoIuiCoeelqOS7o+eggSIpWyLml6XmnJ8iXS5tYXgoKS5kdC5zdHJmdGltZSgiJVktJW0tJWQiKS50b19kaWN0KCkKICAgIG5hbWVzID0gX2xvYWRfbmFtZXMobmFtZXNfY3N2KQogICAgdHJhZGluZ19kYXlzID0gbGlzdChwcm92ZW5hbmNlLmdldCgidHJhZGluZ19kYXlzIikgb3IgW10pCiAgICBlbnRyeSA9IF9udGhfdHJhZGluZ19kYXkodHJhZGluZ19kYXlzLCBhc29mLCAxKSBpZiB0cmFkaW5nX2RheXMgZWxzZSAiIgogICAgZXhpdF9kYXkgPSBfbnRoX3RyYWRpbmdfZGF5KHRyYWRpbmdfZGF5cywgYXNvZiwgNSkgaWYgdHJhZGluZ19kYXlzIGVsc2UgIiIKICAgIHJvd3MgPSBbXQogICAgZm9yIHJhbmssIGluZGV4IGluIGVudW1lcmF0ZShvcmRlciwgc3RhcnQ9MSk6CiAgICAgICAgY29kZSA9IHNlcXVlbmNlX3N0b2NrX2lkc1tpbmRleF0KICAgICAgICByb3dzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgInJhbmsiOiByYW5rLAogICAgICAgICAgICAgICAgInN0b2NrX2NvZGUiOiBjb2RlLAogICAgICAgICAgICAgICAgInN0b2NrX25hbWUiOiBuYW1lcy5nZXQoY29kZSwgIiIpLAogICAgICAgICAgICAgICAgIm1vZGVsX3Njb3JlIjogZmxvYXQoc2NvcmVzW2luZGV4XSksCiAgICAgICAgICAgICAgICAiaG9yaXpvbiI6IEhPUklaT04sCiAgICAgICAgICAgICAgICAiYXNvZl9kYXRlIjogYXNvZiwKICAgICAgICAgICAgICAgICJlbnRyeV9vcGVuX2RhdGUiOiBlbnRyeSwKICAgICAgICAgICAgICAgICJleGl0X29wZW5fZGF0ZSI6IGV4aXRfZGF5LAogICAgICAgICAgICAgICAgImxhYmVsX2Zvcm11bGEiOiBMQUJFTF9GT1JNVUxBLAogICAgICAgICAgICAgICAgImxhc3RfYmFyX2RhdGUiOiBsYXN0X2Jhci5nZXQoY29kZSwgIiIpLAogICAgICAgICAgICAgICAgInN0YWxlX2JhciI6IGJvb2wobGFzdF9iYXIuZ2V0KGNvZGUsICIiKSAhPSBhc29mKSwKICAgICAgICAgICAgICAgICJ3ZWlnaHQiOiB3ZWlnaHRzW3JhbmsgLSAxXSBpZiByYW5rIDw9IHRvcF9rIGVsc2UgIiIsCiAgICAgICAgICAgICAgICAicnVuX21vZGUiOiBwcm92ZW5hbmNlWyJydW5fbW9kZSJdLAogICAgICAgICAgICAgICAgImNvbW1pdCI6IHByb3ZlbmFuY2VbImNvbW1pdCJdLAogICAgICAgICAgICAgICAgInNlZWRzIjogIiwiLmpvaW4oc3RyKHNlZWQpIGZvciBzZWVkIGluIGxvYWRlZCksCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICByYW5raW5nID0gcGQuRGF0YUZyYW1lKHJvd3MpCiAgICBvdXRfY3N2LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICByYW5raW5nLnRvX2NzdihvdXRfY3N2LCBpbmRleD1GYWxzZSwgZW5jb2Rpbmc9InV0Zi04LXNpZyIpCgogICAgb2ZmaWNpYWwgPSByYW5raW5nLmxvY1tyYW5raW5nWyJyYW5rIl0gPD0gdG9wX2ssIFsic3RvY2tfY29kZSIsICJ3ZWlnaHQiXV0ucmVuYW1lKGNvbHVtbnM9eyJzdG9ja19jb2RlIjogInN0b2NrX2lkIn0pCiAgICBvZmZpY2lhbF9wYXRoID0gUGF0aCgiLi9vdXRwdXQvcmVzdWx0LmNzdiIpCiAgICBvZmZpY2lhbF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBvZmZpY2lhbC50b19jc3Yob2ZmaWNpYWxfcGF0aCwgaW5kZXg9RmFsc2UpCiAgICBleGNsdWRlZF9wYXRoID0gb3V0X2Nzdi53aXRoX25hbWUoInZvbGF0aWxpdHlfZXhjbHVkZWQuY3N2IikKICAgIHBkLkRhdGFGcmFtZSh7InN0b2NrX2NvZGUiOiBleGNsdWRlZCwgInN0b2NrX25hbWUiOiBbbmFtZXMuZ2V0KGNvZGUsICIiKSBmb3IgY29kZSBpbiBleGNsdWRlZF19KS50b19jc3YoCiAgICAgICAgZXhjbHVkZWRfcGF0aCwgaW5kZXg9RmFsc2UsIGVuY29kaW5nPSJ1dGYtOC1zaWciCiAgICApCiAgICBwcmludChmIumihOa1i+aXpeacnzoge2Fzb2Z9IikKICAgIHByaW50KGYi5Y+C5LiO5o6S5bqP6IKh56Wo5pWwOiB7bGVuKHJhbmtpbmcpfSIpCiAgICBwcmludChmIue7k+aenOW3suWGmeWFpToge29mZmljaWFsX3BhdGh9IOWSjCB7b3V0X2Nzdn0iKQogICAgcmV0dXJuIHJhbmtpbmcKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW5hbWVzIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByb3ZlbmFuY2UiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgZXhwb3J0X3JhbmtpbmcoYXJncy5uYW1lcywgYXJncy5vdXQsIGFyZ3MucHJvdmVuYW5jZSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "kaggle_driver.py": "..."}
for name, payload in BLOBS.items():
    (KIT / name).write_bytes(base64.b64decode(payload))
    print("wrote", name, (KIT / name).stat().st_size)

if not (REPO / ".git").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "init", str(REPO)])
    subprocess.check_call(["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL])
subprocess.check_call(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", PINNED_COMMIT])
subprocess.check_call(["git", "-C", str(REPO), "checkout", "--force", "--detach", "FETCH_HEAD"])
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("HEAD", head)
if head != PINNED_COMMIT:
    raise SystemExit("checked out commit does not match the pin")


In [ ]:
import importlib.util
import subprocess
import sys

def _have(module):
    return importlib.util.find_spec(module) is not None

# Do not install or upgrade torch. Kaggle's CUDA build must stay as it is.
torch_before = None
if _have("torch"):
    import torch
    torch_before = (torch.__version__, bool(torch.cuda.is_available()), torch.cuda.device_count())
    print("torch before pip", torch_before)

# PyPI has baostock 0.9.4. Its protocol string is 00.9.40; there is no pip release 0.9.40.
# Install only missing scientific packages, with pins that have Python 3.13 wheels.
# only-if-needed avoids replacing the numpy that the existing torch was built against.
pins_if_missing = [
    ("pandas", "pandas==2.3.3"),
    ("sklearn", "scikit-learn==1.7.2"),
    ("joblib", "joblib==1.5.2"),
    ("tensorboardX", "tensorboardX==2.6.4"),
    ("tqdm", "tqdm==4.67.1"),
    ("talib", "ta-lib==0.6.8"),
]
cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade-strategy", "only-if-needed",
    "baostock==0.9.4",
]
for module, pin in pins_if_missing:
    if not _have(module):
        cmd.append(pin)
print("pip install", " ".join(cmd[4:]), flush=True)
subprocess.check_call(cmd)

import baostock
import baostock.common.contants as cons
from importlib.metadata import version
if version("baostock") != "0.9.4":
    raise SystemExit("baostock dist is %s, expected 0.9.4" % version("baostock"))
if cons.BAOSTOCK_CLIENT_VERSION != "00.9.40":
    raise SystemExit("unexpected BaoStock protocol client %s" % cons.BAOSTOCK_CLIENT_VERSION)
print("baostock", version("baostock"), "protocol", cons.BAOSTOCK_CLIENT_VERSION)

if torch_before is not None:
    import importlib
    importlib.reload(torch)
    torch_after = (torch.__version__, bool(torch.cuda.is_available()), torch.cuda.device_count())
    print("torch after pip", torch_after)
    if torch_after != torch_before:
        raise SystemExit("pip changed the existing torch install: %s -> %s" % (torch_before, torch_after))


In [ ]:
import subprocess
import sys
from pathlib import Path

command = [
    sys.executable, "-u", str(KIT / "kaggle_driver.py"),
    "--repo", str(REPO),
    "--output-dir", str(OUTPUT),
    "--mode", RUN_MODE,
]
if RESUME_COMPLETED_SEEDS:
    command.append("--resume")
if USE_OFFLINE_SNAPSHOT:
    command.extend([
        "--offline-panel", OFFLINE_PANEL,
        "--offline-names", OFFLINE_NAMES,
        "--offline-manifest", OFFLINE_MANIFEST,
    ])
if ALLOW_STALE_SNAPSHOT:
    command.append("--allow-stale-snapshot")
print(" ".join(command), flush=True)
subprocess.check_call(command)


In [ ]:
from pathlib import Path
import pandas as pd
ranking_path = Path("/kaggle/working/hs300_rank_output/ranking_full.csv")
result_path = Path("/kaggle/working/hs300_rank_output/result.csv")
pass_path = Path("/kaggle/working/hs300_rank_output/VERIFY_PASS.txt")
if RUN_MODE != "full":
    print("No final ranking is displayed because RUN_MODE is", RUN_MODE)
elif not pass_path.is_file():
    raise SystemExit("VERIFY_PASS.txt is missing; do not use any CSV from this run as the ranking")
else:
    ranking = pd.read_csv(ranking_path, dtype={"stock_code": str}, encoding="utf-8-sig")
    result = pd.read_csv(result_path, dtype={"stock_id": str})
    print("asof", ranking["asof_date"].iloc[0])
    print("horizon", ranking["horizon"].iloc[0])
    print("entry", ranking["entry_open_date"].iloc[0], "exit", ranking["exit_open_date"].iloc[0])
    print(result.to_string(index=False))
    print(ranking.head(15).to_string(index=False))
